In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from joblib import Parallel, delayed
from datatools import *
import matplotlib.pyplot as plt
import seaborn as sns

### 策略

In [3]:
start_date = '2019-01-01'
end_date = '2026-06-30'
allstocks = pd.read_csv('C:/Users/User/OneDrive - CUHK-Shenzhen/data/allstock.csv')
allstocks = allstocks[~allstocks['ts_code'].str.contains('BJ')].ts_code.tolist()
load_start = (pd.Timestamp(start_date) - pd.DateOffset(years=1)).strftime('%Y-%m-%d')
price = get_price(codes=allstocks, start_date=load_start, end_date=end_date)
price['daily_return'] = price['close']/price['pre_close'] - 1
price['next_open'] = price.groupby('ts_code')['open'].shift(-1)
price = price.sort_values(by=['ts_code', 'trade_date']).reset_index(drop=True)

In [9]:
st = get_st(codes=allstocks, start_date=load_start, end_date=end_date)

In [12]:
df2_keys_set = set(zip(st ['trade_date'], st['ts_code']))
df1_keys_tuples = list(zip(price['trade_date'], price['ts_code']))
mask = [tup not in df2_keys_set for tup in df1_keys_tuples]
price = price[mask]

In [13]:
len(price)

8857566

### rsrs

In [ ]:
# ── 构建矩阵 ──────────────────────────────────────────────────────────────
dates = np.sort(price['trade_date'].unique())
universe = np.sort(price['ts_code'].unique())

high_pivot = price.pivot(index='trade_date', columns='ts_code', values='high').reindex(index=dates, columns=universe)
low_pivot = price.pivot(index='trade_date', columns='ts_code', values='low').reindex(index=dates, columns=universe)

n_dates, n_stocks = len(dates), len(universe)
print(f'矩阵: {n_dates} 交易日 × {n_stocks} 只股票')
N=18
M=600
cov_xy = high_pivot.rolling(N, min_periods=2).cov(low_pivot)
var_x = low_pivot.rolling(N, min_periods=2).var()
var_y = high_pivot.rolling(N, min_periods=2).var()
slope = cov_xy / var_x
r2 = (cov_xy ** 2) / (var_x * var_y)
slope_zscore = (slope - slope.rolling(M, min_periods=2).mean()) / slope.rolling(M, min_periods=2).std()
rsrs = slope*r2*slope_zscore
rsrs = rsrs.loc[start_date:end_date]
rsrs = rsrs.stack().reset_index().rename(columns={0:'rsrs'})

矩阵: 4955 交易日 × 5489 只股票


### 最低点&MACD&BOLL策略

In [7]:
price['row_idx'] = price.groupby('ts_code').cumcount()
g = price.groupby('ts_code')

# 条件1: 过去5天最低收盘价 == 过去180天最低收盘价
# (等价于过去5天内某天收盘价创了180天新低)
price['min_close_15'] = g['close'].rolling(15).min().reset_index(level=0, drop=True)
price['min_close_180'] = g['close'].rolling(180).min().reset_index(level=0, drop=True)
price['cond1'] = price['min_close_15'] == price['min_close_180']

# 条件2: 过去5天最低价 < 布林带下界 (默认20日, 2倍标准差)
price['ma20'] = g['close'].rolling(20).mean().reset_index(level=0, drop=True)
price['std20'] = g['close'].rolling(20).std().reset_index(level=0, drop=True)
price['lower_band'] = price['ma20'] - 2 * price['std20']

price['low_below_band'] = price['low'] < price['lower_band']
price['cond2'] = g['low_below_band'].rolling(5).max().reset_index(level=0, drop=True) > 0

# 条件3: MACD由绿转红 (柱状值今日大于0，昨日小于等于0)
price['ema12'] = g['close'].ewm(span=12, adjust=False).mean().reset_index(level=0, drop=True)
price['ema26'] = g['close'].ewm(span=26, adjust=False).mean().reset_index(level=0, drop=True)
price['dif'] = price['ema12'] - price['ema26']
price['dea'] = g['dif'].ewm(span=9, adjust=False).mean().reset_index(level=0, drop=True)
price['macd_hist'] = (price['dif'] - price['dea']) * 2
price['cond3'] = (price['macd_hist'] > 0) & (g['macd_hist'].shift(1) < 0)

price['cond4'] = g['macd_hist'].shift(1).rolling(15).max().reset_index(level=0, drop=True) < 0
price['signal'] = price['cond1'] & price['cond2'] & price['cond3'] & price['cond4']

In [8]:
df_res = price[price['signal']]
df_res = df_res.merge(
    price[['ts_code', 'row_idx', 'trade_date', 'next_open']], 
    on=['ts_code', 'row_idx'], suffixes=('', '_buy')
).rename(columns={'trade_date_buy': 'buy_date', 'next_open_buy': 'buy_price'})

df_res['sell_idx'] = df_res['row_idx'] + 20
trade_records = df_res.merge(
    price[['ts_code', 'row_idx', 'trade_date', 'next_open']], 
    left_on=['ts_code', 'sell_idx'], 
    right_on=['ts_code', 'row_idx'],
    suffixes=('', '_sell')
).rename(columns={'trade_date_sell': 'sell_date', 'next_open_sell': 'sell_price'})

trade_records = trade_records[['ts_code', 'buy_date', 'sell_date', 'buy_price', 'sell_price']].dropna().reset_index(drop=True)


In [ ]:
trade_records

,ts_code,buy_date,sell_date,buy_price,sell_price
0,000001.SZ,2022-03-21,2022-04-20,1632.94198,1765.00994
1,000002.SZ,2022-03-22,2022-04-21,2872.30805,3108.27670
2,000004.SZ,2018-09-25,2018-11-20,64.61760,68.72224
3,000004.SZ,2022-10-17,2022-11-14,35.92576,38.77056
4,000004.SZ,2025-04-15,2025-05-19,41.24960,35.35680
...,...,...,...,...,...
5929,688819.SH,2022-05-06,2022-06-06,28.27790,35.82950
5930,688819.SH,2024-07-11,2024-08-08,25.71640,24.03784
5931,688981.SH,2021-07-27,2021-08-24,62.95000,60.06000
5932,688981.SH,2022-05-09,2022-06-07,39.53000,45.17000


: 

In [22]:
def strategy_metrics(daily_return_df, trade_records, annual_trading_days=252, cost=0.001):
    all_dates = pd.DataFrame(daily_return_df['trade_date'].sort_values().unique(), columns=['trade_date'])
    holding_df = daily_return_df.merge(trade_records, on='ts_code')
    holding_df = holding_df[(holding_df['trade_date'] > holding_df['buy_date']) & 
                            (holding_df['trade_date'] <= holding_df['sell_date'])]

    daily_strategy_ret = holding_df.groupby('trade_date')['daily_return'].mean().reset_index()

    net_value_df = all_dates.merge(daily_strategy_ret, on='trade_date', how='left')
    net_value_df = net_value_df.fillna(0)
    net_value_df['daily_return'] -= cost
    net_value_df['cum_nav'] = (1 + net_value_df['daily_return']).cumprod()
    return_annual = net_value_df['cum_nav'].iloc[-1] ** (annual_trading_days/len(net_value_df)) - 1
    volatility_annual = net_value_df['daily_return'].std() * np.sqrt(annual_trading_days)
    sharpe_ratio = (return_annual - 0.04) / volatility_annual
    
    drawdown = net_value_df['cum_nav'] / net_value_df['cum_nav'].expanding().max() - 1
    max_drawdown = drawdown.min()

    win_rate = (trade_records['sell_price'] > trade_records['buy_price']).sum() / len(trade_records)
    
    return {'sharpe_ratio': sharpe_ratio, 'max_drawdown': max_drawdown, 'win_rate': win_rate, 'return_annual':return_annual}

In [ ]:
alpha = strategy_metrics(price, trade_records)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
